# Model training and tuning

This notebook trains and tunes remaining useful life (RUL) regressors for NASA C-MAPSS FD001. Mean and linear-regression baselines are compared with Elastic Net, Random Forest, and XGBoost on two feature sets: raw sensor readings and the engineered degradation features. All validation uses five-fold cross-validation grouped by engine, so every fold holds out complete engines. Only the training trajectories are used here.

## Experimental setup

Training targets are RUL capped at 125 cycles. The sensors are the 14 informative sensors identified in the exploratory analysis; constant and near-constant sensors are excluded. Both feature sets include the current cycle number:

- **Raw:** cycle number and the 14 sensor readings.
- **Engineered:** raw features plus rolling means, standard deviations, and least-squares slopes of the regime-normalised sensors over 5, 10, and 20 cycle windows, and the health index.

The engineered features depend on statistics fitted to training data (regime normalisation and the health-index model). These are fitted on each fold's training engines only and then applied to the held-out engines, so no information from validation engines enters feature construction.

In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import ParameterGrid, ParameterSampler

from rul.data import USEFUL_SENSORS, add_rul, load_train_fd
from rul.evaluate import evaluate
from rul.features import feature_groups, make_features
from rul.models import build_model
from rul.split import engine_folds

RUL_CAP = 125
SEED = 42
WINDOWS = (5, 10, 20)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pd.set_option("display.precision", 3)

In [2]:
train = load_train_fd("FD001")
train = add_rul(train).rename(columns={"rul": "rul_true"})
train["rul"] = train["rul_true"].clip(upper=RUL_CAP)

groups = feature_groups(USEFUL_SENSORS, WINDOWS)
FEATURE_SETS = {
    "raw": groups["raw"],
    "engineered": groups["raw"] + groups["rolling"] + groups["slope"] + groups["health"],
}

# Build features once per fold; the feature scaler is fitted on that fold's training engines only.
fold_data = []
for train_idx, val_idx in engine_folds(train, n_splits=5):
    fold_train, scaler = make_features(train.iloc[train_idx], USEFUL_SENSORS, WINDOWS)
    fold_val, _ = make_features(train.iloc[val_idx], USEFUL_SENSORS, WINDOWS, scaler=scaler)
    fold_data.append((fold_train, fold_val))

print(f"Training rows: {len(train):,}; engines: {train.unit.nunique()}")
print({name: len(cols) for name, cols in FEATURE_SETS.items()}, "features")

Training rows: 20,631; engines: 100
{'raw': 15, 'engineered': 142} features


In [3]:
def cross_validate(model, features, target="rul", truth="rul"):
    """Fit on each fold's training engines; return one row of metrics per held-out fold."""
    rows = []
    for fold_train, fold_val in fold_data:
        fitted = clone(model).fit(fold_train[features], fold_train[target])
        prediction = fitted.predict(fold_val[features])
        # Where true RUL <= cap the capped and true labels agree, so this column compares any rows.
        near = (fold_val.rul_true <= RUL_CAP).to_numpy()
        error_near = prediction[near] - fold_val.rul_true.to_numpy()[near]
        rows.append({**evaluate(fold_val[truth], prediction),
                     "rmse_le_cap": (error_near**2).mean() ** 0.5})
    return pd.DataFrame(rows)


def summarise(fold_scores):
    """Mean and standard deviation across folds of RMSE and MAE, plus other fold means."""
    return {
        "rmse_mean": fold_scores.rmse.mean(), "rmse_std": fold_scores.rmse.std(),
        "mae_mean": fold_scores.mae.mean(), "mae_std": fold_scores.mae.std(),
        "nasa_score_mean": fold_scores.nasa_score.mean(),
        "rmse_late_mean": fold_scores.rmse_late.mean(),
        "rmse_le_cap_mean": fold_scores.rmse_le_cap.mean(),
    }

## Baselines

The mean predictor gives a constant-prediction reference, and ordinary linear regression on the raw features shows how much a simple linear relationship can explain. Every tuned model should improve on both. Linear regression is also fitted on the engineered features, as an unregularised reference for Elastic Net on the same inputs. Metrics are computed over all held-out cycles against the capped target. `rmse_late_mean` restricts RMSE to cycles with true RUL of at most 50, the stage where accurate predictions matter most for maintenance, and `rmse_le_cap_mean` restricts it to cycles with true RUL of at most 125, where capped and true labels coincide.

In [4]:
baselines = pd.DataFrame([
    {"model": "Mean baseline", "features": "raw",
     **summarise(cross_validate(DummyRegressor(), FEATURE_SETS["raw"]))},
    *({"model": "Linear Regression", "features": name,
       **summarise(cross_validate(LinearRegression(), features))}
      for name, features in FEATURE_SETS.items()),
]).set_index("model")
baselines

,features,rmse_mean,rmse_std,mae_mean,mae_std,nasa_score_mean,rmse_late_mean,rmse_le_cap_mean
model,,,,,,,,
Mean baseline,raw,41.673,0.004,36.965,0.003,1.271e+06,63.557,43.759
Linear Regression,raw,20.119,1.544,16.368,1.287,3.193e+04,21.705,20.790
Linear Regression,engineered,17.066,0.781,13.718,0.630,1.983e+04,16.886,17.646


## Hyperparameter search

Each model is tuned separately on each feature set, using the same five engine-grouped folds, and the configuration with the lowest mean RMSE is selected. Elastic Net is cheap to fit, so its full grid is searched; it is solved by coordinate descent on a precomputed Gram matrix with up to 100,000 iterations, enough for every configuration to converge. Random Forest and XGBoost use a random sample of configurations from their grids, drawn with seed 42.

| Model | Search space | Configurations |
|---|---|---|
| Elastic Net | `alpha` ∈ {0.001, 0.01, 0.1, 0.3, 1.0}, `l1_ratio` ∈ {0.2, 0.5, 0.8, 1.0} | 20 (full grid) |
| Random Forest | `n_estimators` ∈ {100, 200}, `max_depth` ∈ {8, 12, None}, `min_samples_leaf` ∈ {1, 5, 10}, `max_features` ∈ {0.3, 0.5, 1.0} | 6 (random) |
| XGBoost | `n_estimators` ∈ {200, 400}, `learning_rate` ∈ {0.03, 0.05, 0.1}, `max_depth` ∈ {3, 4, 6}, `subsample` ∈ {0.8, 1.0}, `colsample_bytree` ∈ {0.5, 0.8, 1.0}, `reg_lambda` ∈ {1, 5} | 8 (random) |

`alpha` sets the overall strength of the Elastic Net penalty and `l1_ratio` the share of it that is L1 (sparsity) rather than L2 (shrinkage); `l1_ratio` = 1 is the pure L1 (Lasso) end of the family. For Random Forest, depth, leaf size, and the fraction of features considered at each split control how closely individual trees fit the data and how correlated they are. For XGBoost, the learning rate and number of rounds trade off together, depth limits interaction order, row and column subsampling add randomness, and `reg_lambda` is the L2 penalty on leaf weights.

In [5]:
SEARCH_SPACES = {
    "elastic_net": list(ParameterGrid({
        "alpha": [0.001, 0.01, 0.1, 0.3, 1.0], "l1_ratio": [0.2, 0.5, 0.8, 1.0], "max_iter": [100_000],
        "precompute": [True],
    })),
    "random_forest": list(ParameterSampler({
        "n_estimators": [100, 200], "max_depth": [8, 12, None], "min_samples_leaf": [1, 5, 10],
        "max_features": [0.3, 0.5, 1.0],
    }, n_iter=6, random_state=SEED)),
    "xgboost": list(ParameterSampler({
        "n_estimators": [200, 400], "learning_rate": [0.03, 0.05, 0.1], "max_depth": [3, 4, 6],
        "subsample": [0.8, 1.0], "colsample_bytree": [0.5, 0.8, 1.0], "reg_lambda": [1, 5],
    }, n_iter=8, random_state=SEED)),
}
PARALLEL = {"random_forest": {"n_jobs": -1}, "xgboost": {"n_jobs": -1}, "elastic_net": {}}

search_rows = []
for feature_set, features in FEATURE_SETS.items():
    for name, space in SEARCH_SPACES.items():
        for params in space:
            model = build_model(name, **params, **PARALLEL[name])
            scores = summarise(cross_validate(model, features))
            search_rows.append({"features": feature_set, "model": name, "params": params, **scores})
search = pd.DataFrame(search_rows)
search.sort_values(["features", "model", "rmse_mean"]).drop(columns="params")

,features,model,rmse_mean,rmse_std,mae_mean,mae_std,nasa_score_mean,rmse_late_mean,rmse_le_cap_mean
36,engineered,elastic_net,17.063,0.796,13.719,0.639,19798.819,16.859,17.632
37,engineered,elastic_net,17.065,0.785,13.718,0.633,19815.320,16.879,17.642
41,engineered,elastic_net,17.066,0.826,13.726,0.659,19777.957,16.828,17.618
35,engineered,elastic_net,17.067,0.808,13.724,0.647,19797.689,16.841,17.626
34,engineered,elastic_net,17.075,0.819,13.731,0.655,19808.983,16.833,17.624
...,...,...,...,...,...,...,...,...,...
29,raw,xgboost,16.529,0.899,11.417,0.664,25370.030,15.286,18.453
26,raw,xgboost,16.580,0.937,11.425,0.677,26228.893,15.232,18.485
33,raw,xgboost,16.654,0.813,11.719,0.719,25157.177,15.803,18.513
28,raw,xgboost,16.832,0.931,11.520,0.739,28861.077,15.263,18.792


## Effect of the RUL cap

Capping the target assumes engines show no measurable degradation early in life, so the model is not asked to distinguish, for example, 300 remaining cycles from 200. To test this, the best configuration is retrained on the uncapped RUL with the same folds. A fair comparison needs both models scored against the same labels, so both are evaluated against the true, uncapped RUL.

In [6]:
best = search.loc[search.groupby(["features", "model"]).rmse_mean.idxmin()]
best_row = best.loc[best.rmse_mean.idxmin()]
best_model = build_model(best_row.model, **best_row.params, **PARALLEL[best_row.model])
best_features = FEATURE_SETS[best_row.features]
print(f"Best configuration: {best_row.model} on {best_row.features} features")

cap_rows = pd.DataFrame({
    (best_row.model, best_row.features, f"capped at {RUL_CAP}", "true RUL"):
        summarise(cross_validate(best_model, best_features, target="rul", truth="rul_true")),
    (best_row.model, best_row.features, "uncapped", "true RUL"):
        summarise(cross_validate(best_model, best_features, target="rul_true", truth="rul_true")),
}).T

Best configuration: xgboost on engineered features


## Cross-validation results

The table reports, for the baselines and for each model's selected configuration on each feature set, the mean and standard deviation of RMSE and MAE across the five folds. All models are trained on the capped target and scored against it, except the final two rows: the best configuration trained with and without the cap, both scored against the true RUL. Because those two rows use different labels from the rest, their overall RMSE and MAE are comparable with each other but not with the rows above. `rmse_le_cap_mean` is computed on the same cycles and labels for every row.

These are model-selection scores: the same folds are used to choose each configuration and to report its score, so the scores of the selected configurations are slightly optimistic. Grouping by engine prevents leakage between engines, and the small search limits the optimism, but neither removes it. The held-out test set, scored separately with the frozen configurations, gives the independent estimate.

In [7]:
capped = {"training target": f"capped at {RUL_CAP}", "scored against": "capped RUL"}
cv_table = pd.concat([
    baselines.assign(**capped),
    best.set_index("model").drop(columns="params").assign(**capped),
]).sort_values("features", ascending=False, kind="stable").set_index(["features", "training target", "scored against"], append=True)
cv_table = pd.concat([cv_table, cap_rows]).rename_axis(
    ["model", "features", "training target", "scored against"])
cv_table

,,,,rmse_mean,rmse_std,mae_mean,mae_std,nasa_score_mean,rmse_late_mean,rmse_le_cap_mean
model,features,training target,scored against,,,,,,,
Mean baseline,raw,capped at 125,capped RUL,41.673,0.004,36.965,0.003,1.271e+06,63.557,43.759
Linear Regression,raw,capped at 125,capped RUL,20.119,1.544,16.368,1.287,3.193e+04,21.705,20.790
elastic_net,raw,capped at 125,capped RUL,20.119,1.542,16.370,1.287,3.191e+04,21.718,20.785
random_forest,raw,capped at 125,capped RUL,16.433,0.770,11.465,0.711,2.440e+04,15.401,18.289
xgboost,raw,capped at 125,capped RUL,16.421,0.868,11.382,0.697,2.465e+04,15.232,18.277
Linear Regression,engineered,capped at 125,capped RUL,17.066,0.781,13.718,0.630,1.983e+04,16.886,17.646
elastic_net,engineered,capped at 125,capped RUL,17.063,0.796,13.719,0.639,1.980e+04,16.859,17.632
random_forest,engineered,capped at 125,capped RUL,14.420,0.482,9.875,0.560,1.582e+04,10.983,15.761
xgboost,engineered,capped at 125,capped RUL,13.988,0.461,9.805,0.415,1.450e+04,9.883,15.093


Engineered features improve every model: XGBoost RMSE falls from 16.42 to 13.99 cycles, Random Forest from 16.43 to 14.42, and Elastic Net from 20.12 to 17.06. The tree ensembles clearly outperform the linear models on both feature sets. A linear model is linear in its inputs, so it can only follow the nonlinear, accelerating degradation to the extent that the features already encode it; the rolling statistics, slopes, and health index supply part of that structure, which is why the linear models gain from them, while trees can learn the remaining nonlinearity directly. XGBoost on engineered features has the lowest mean RMSE and also the lowest RMSE on late-life cycles, although Random Forest is within one fold standard deviation of it on overall RMSE.

All tuned models beat the mean baseline by a wide margin. Elastic Net ties with unregularised linear regression on both feature sets. On raw features its RMSE is 0.0006 cycles lower and its MAE 0.0015 cycles higher; on engineered features its RMSE is 0.003 cycles lower (17.063 against 17.066) and its MAE 0.0003 cycles higher. Every difference is far inside the fold standard deviations of 0.8 to 1.5 cycles. With at most 142 features and more than 16,000 training rows per fold, the linear models are limited by bias rather than variance, so the penalty has almost nothing to remove. On engineered features the error rises steadily as `alpha` grows (17.75 at `alpha` = 0.1, 19.39 at `alpha` = 1.0, both pure L1), and the search selects the smallest value in the grid, `alpha` = 0.001. Smaller values would only move Elastic Net closer to ordinary least squares, whose score is the linear-regression row above. The penalty does not improve predictive accuracy here; the gain on engineered features comes from the features themselves.

For the RUL cap, the evidence depends on which cycles are scored. On cycles with true RUL of at most 125, the capped model is far more accurate (RMSE 15.09 against 25.99). Over all cycles the uncapped model scores better (38.26 against 47.47). The capped model never sees a training target above 125, so for engines early in life, whose true RUL reaches several hundred cycles, its predictions stay close to 125 and its errors are large. Capping changes the training target, not the range of the predictions. It concentrates the model on the part of life where degradation is visible, trading accuracy on early-life cycles, where predictions carry little practical value, for accuracy close to failure, where maintenance decisions are made. The cap is therefore kept for all tuned models.

In [8]:
BEST_PARAMS_PATH = ROOT / "results" / "best_params.json"
best_params = {
    "selection": "lowest mean RMSE, five-fold GroupKFold by unit on training engines",
    "target_cap": RUL_CAP,
    "sensors": USEFUL_SENSORS,
    "windows": list(WINDOWS),
    "selected": {"model": best_row.model, "features": best_row.features},
    "models": {
        f"{row.model}/{row.features}": {
            "params": row.params,
            "rmse_mean": round(row.rmse_mean, 3), "rmse_std": round(row.rmse_std, 3),
            "mae_mean": round(row.mae_mean, 3), "mae_std": round(row.mae_std, 3),
        }
        for row in best.itertuples()
    },
}
BEST_PARAMS_PATH.write_text(json.dumps(best_params, indent=2) + "\n")
cv_table.to_csv(ROOT / "results" / "models_cv.csv")
search.to_csv(ROOT / "results" / "models_search.csv", index=False)
print(json.dumps(best_params["selected"]))

{"model": "xgboost", "features": "engineered"}
